# TP2 — Generation : des passages pertinents à la réponse

**Objectif de l'après-midi** : ajouter le **G** de RAG. Les k chunks trouvés au TP1 sont donnés à un LLM,
avec la question, pour qu'il rédige une réponse **appuyée sur les documents et vérifiable**.

```
question ──► search (TP1) ──► k chunks ──┐
                                          ├──► prompt ──► LLM ──► réponse + sources
consignes (system) + question ────────────┘
```

Plan :
1. Appeler un LLM via une API
2. Sans RAG : le LLM seul
3. Avec RAG
4. Expériences : nombre de chunks, petit ou gros modèle
5. Exercices
6. En dehors du notebook : la ligne de commande

Prérequis : l'index du TP1 (`index/`) et la clé d'API dans `.env` (voir `README.md`).

In [ ]:
import time

from dotenv import load_dotenv
from litellm import completion

import settings
from rag import load_index, search, build_prompt, generate, answer

load_dotenv()  # charge HF_TOKEN depuis .env
chunks, embeddings = load_index()  # l'index construit au TP1 (sinon : python rag.py)
print(len(chunks), "chunks,", embeddings.shape[1], "dimensions")

## 1. Appeler un LLM via une API

Un LLM de plusieurs milliards de paramètres tourne sur des GPU distants : on l'appelle via une **API HTTP**.
Tous les fournisseurs (OpenAI, Mistral, Anthropic, Hugging Face, …) ont adopté le même format de conversation :
une liste de **messages**, chacun avec un rôle :

- `system` : les consignes données au modèle (ton, format, limites) ;
- `user` : ce que dit l'utilisateur ;
- `assistant` : ce que répond le modèle.

La librairie **LiteLLM** expose une seule fonction, `completion()`, quel que soit le fournisseur.
Ici on appelle des **modèles ouverts** hébergés par Hugging Face (*Inference Providers*) :
le modèle est choisi dans `settings.GENERATION_MODEL`, la clé d'API est lue dans `.env`.

In [ ]:
messages = [
    {"role": "system", "content": "Tu es un assistant concis. Réponds en deux phrases maximum."},
    {"role": "user", "content": "Qu'est-ce qu'un capteur à ultrasons ?"},
]

reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0)

print(reponse.choices[0].message.content)
print("\nusage :", reponse.usage)

`usage` donne le nombre de tokens **envoyés** (`prompt_tokens`) et **générés** (`completion_tokens`) : c'est ce qui est facturé,
et c'est ce qui doit tenir dans la fenêtre de contexte du modèle.

La **température** règle le hasard dans le choix de chaque token : 0 = le token le plus probable à chaque fois
(réponses stables, ce qu'on veut pour du factuel) ; élevée = plus de variété (et plus d'erreurs).

In [ ]:
messages = [{"role": "user", "content": "Propose un nom original pour un robot explorateur. Réponds par le nom seul."}]

for temperature in (0.0, 0.0, 1.5, 1.5):
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=temperature, max_tokens=20)
    print(f"température {temperature} → {reponse.choices[0].message.content.strip()}")

## 2. Sans RAG : le LLM seul

Posons une question précise sur nos datasheets, **sans lui donner les documents**.

In [ ]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

reponse = completion(model=settings.GENERATION_MODEL, messages=[{"role": "user", "content": question}], temperature=0)
print(reponse.choices[0].message.content)

**À observer**

Le modèle répond avec assurance… mais a-t-il raison ? Il n'a que ce qu'il a mémorisé pendant son entraînement
(peut-être cette datasheet, peut-être pas, peut-être une version différente). Rien ne permet de le vérifier :
c'est le problème des **hallucinations**. Vérifiez dans `data/bme280.pdf` (chapitre "Electrical characteristics").

## 3. Avec RAG

Trois étapes :
1. **chercher** les chunks pertinents (TP1 : `search`) ;
2. **construire le prompt** : consignes + extraits + question (`build_prompt`) ;
3. **générer** la réponse (`generate`).

Regardons d'abord le prompt réellement envoyé au modèle.

In [ ]:
extraits = search(question, chunks, embeddings, k=settings.TOP_K)   # 1. retrieval
messages = build_prompt(question, extraits)                         # 2. prompt

print("=== system ===")
print(messages[0]["content"])
print("\n=== user (début) ===")
print(messages[1]["content"][:1500], "\n[...]")

In [ ]:
print(generate(messages))                                           # 3. generation

**À observer**

- La connaissance n'est plus dans le modèle : elle est **dans le prompt**. C'est ça, "augmented".
- La réponse cite `[fichier, page]` : on peut aller **vérifier**. C'est le principal intérêt du RAG en entreprise.
- Le prompt système impose : répondre uniquement à partir des extraits, dire quand l'information n'y est pas, citer.

`answer` dans `rag.py` enchaîne les trois étapes et renvoie la réponse **et** les chunks utilisés.

In [ ]:
questions = [
    "Quelle est la plage de tension d'alimentation (VSYS) du Raspberry Pi Pico ?",
    "Quelle est la fréquence maximale du processeur de l'ESP32 ?",
    "Quelle est la précision du LM35 à 25 °C ?",
]

for q in questions:
    reponse, utilises = answer(q, chunks, embeddings)
    print("###", q)
    print(reponse)
    print("chunks utilisés :", ", ".join(f"{c['source']} p.{c['page']}" for c in utilises))
    print()

Et si la réponse n'est **pas** dans les documents ? Un bon RAG doit le dire plutôt qu'inventer.

In [ ]:
reponse, utilises = answer("Quel est le prix du Raspberry Pi Pico ?", chunks, embeddings)
print(reponse)

## 4. Expériences

### 4.a Combien de chunks donner au modèle ?

`k` (dans `settings.TOP_K`) est un compromis : plus de chunks = plus de chances que la réponse y soit,
mais plus de tokens (coût, latence), plus de bruit pour le modèle, et une fenêtre de contexte qui a une limite.

In [ ]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

for k in (1, 5, 15):
    messages = build_prompt(question, search(question, chunks, embeddings, k=k))
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0, max_tokens=200)
    print(f"k={k:<3} tokens envoyés : {reponse.usage.prompt_tokens:<6} → {reponse.choices[0].message.content[:160]!r}")

### 4.b Petit modèle + RAG ou gros modèle seul ?

Comparons un modèle de 7 milliards de paramètres et un de 72 milliards (dix fois plus cher et plus lent),
avec et sans RAG, sur la même question.

In [ ]:
essais = [
    ("petit modèle, sans RAG", settings.GENERATION_MODEL, False),
    ("petit modèle + RAG", settings.GENERATION_MODEL, True),
    ("gros modèle, sans RAG", settings.GENERATION_MODEL_LARGE, False),
    ("gros modèle + RAG", settings.GENERATION_MODEL_LARGE, True),
]

for nom, modele, avec_rag in essais:
    if avec_rag:
        messages = build_prompt(question, search(question, chunks, embeddings))
    else:
        messages = [{"role": "user", "content": question}]
    debut = time.time()
    reponse = completion(model=modele, messages=messages, temperature=0, max_tokens=200)
    duree = time.time() - debut
    print(f"### {nom} — {duree:.1f} s, {reponse.usage.total_tokens} tokens")
    print(reponse.choices[0].message.content.strip(), "\n")

**À observer**

Un petit modèle **bien alimenté en contexte** fait souvent aussi bien qu'un gros modèle livré à sa mémoire —
pour une fraction du coût, de la latence et de l'énergie. C'est l'argument de **frugalité** du RAG :
la connaissance est dans les documents (gratuits à lire), l'intelligence du modèle ne sert qu'à la restituer.

## 5. Exercices

**Exercice 1 — le prompt système.** Modifiez `SYSTEM_PROMPT` dans `rag.py` : format de réponse (liste, tableau),
unités, longueur, comportement quand la réponse est absente, citation en fin de phrase ou en fin de réponse…
Observez l'effet sur les questions ci-dessus.

**Exercice 2 — recherche hybride dans le RAG.** Copiez votre `search_hybrid` du TP1 et utilisez-la à la place
de `search` dans `answer`. Les questions sur des identifiants exacts (`ctrl_meas`, `GPIO25`…) s'améliorent-elles ?

**Exercice 3 — réécriture de la question.** Les datasheets sont en anglais et les questions en français.
Avant la recherche, demandez au LLM de traduire la question en une requête technique anglaise, puis cherchez avec cette
requête. Comparez les chunks trouvés avec et sans réécriture. (Le LLM sert ici *aussi* à améliorer le retrieval.)

**Bonus — conversation.** Gardez l'historique des messages entre deux questions (liste `messages` qui grandit :
`user`, `assistant`, `user`, …) pour permettre "et pour l'ESP32 ?" après une question sur le Pico.
Que faut-il faire de l'historique quand il devient trop long pour la fenêtre de contexte ?

In [ ]:
def rewrite_query(question):
    # Exercice 3 : renvoyer une requête de recherche en anglais technique, générée par le LLM
    ...


# afficher(search(rewrite_query(question), chunks, embeddings))

## 6. En dehors du notebook

Le même `rag.py` s'utilise directement :

```bash
poetry run python rag.py "Quelle est la tension d'alimentation du HC-SR04 ?"
```

## Bilan

```
PDF ──► chunks ──► embeddings ──► index                     (indexation, TP1)
question ──► search ──► k chunks ──► prompt ──► LLM ──► réponse + sources   (TP1 + TP2)
```

Toutes les briques ont des réglages (découpage, modèle d'embeddings, k, hybride, prompt, modèle de génération…).
Pour choisir sans se fier à l'impression laissée par trois questions, il faut **mesurer** : c'est le TP3.